# Machine Learning

In [1]:
# === Instalações (se necessário) ===
%pip install xgboost --quiet
%pip install lightgbm --quiet
%pip install -U scikit-learn --quiet
%pip install scikit-survival --quiet
%pip install seaborn --quiet
%pip install --upgrade lifelines --quiet

# === Split ===
from sklearn.model_selection import train_test_split

# === Survival (scikit-survival) ===
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw, integrated_brier_score, brier_score
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

# === Encoders (você já usa) ===
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# === Métricas auxiliares de "tempo previsto" (quando existir) ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
import numpy as np
import pandas as pd

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# === Additional imports for Phase 3 helpers ===
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from lifelines import CoxPHFitter
from sklearn.model_selection import StratifiedKFold
from sksurv.ensemble import RandomSurvivalForest

# === PHASE 3: CONSOLIDATED HELPER FUNCTIONS ===
# All functions defined once, in dependency order.
# This ensures single source of truth for all model development logic.

# ============================================================================
# SECTION 1: Data Splitting & Encoding Functions
# ============================================================================

def split_dev_test(df, dev_ratio=0.85, test_ratio=0.15, random_state=19, stratify_col="event"):
    """
    Divide o dataframe em dois conjuntos: desenvolvimento e teste externo.
    
    O conjunto de desenvolvimento (dev) será usado para todo o desenvolvimento do modelo
    (preprocessamento, seleção de modelos, tuning de hiperparâmetros, validação cruzada).
    O conjunto teste externo é bloqueado até a avaliação final.
    
    Parâmetros:
      dev_ratio: float - Proporção de dados para desenvolvimento (default=0.85)
      test_ratio: float - Proporção de dados para teste externo (default=0.15)
      random_state: int - Seed para reprodutibilidade
      stratify_col: str - Coluna para estratificação
    
    Retorno:
      (df_dev, df_test_external) - DataFrames com índices resetados
    """
    df_aux = df.copy().reset_index(drop=True)
    
    assert abs(dev_ratio + test_ratio - 1.0) < 1e-6, \
        f"Proporções devem somar 1.0, obtido: {dev_ratio + test_ratio}"
    
    strat = df_aux[stratify_col] if stratify_col in df_aux.columns else None
    
    df_dev, df_test_ext = train_test_split(
        df_aux,
        test_size=test_ratio,
        random_state=random_state,
        stratify=strat
    )
    
    return (
        df_dev.reset_index(drop=True), 
        df_test_ext.reset_index(drop=True)
    )


def fit_encoders_train(
    df_train,
    ordinal_col="ECGRUP",
    ordinal_order=("I", "II", "III", "IV"),
    onehot_cols=("TOPO", "MORFO_CAT"),
    drop_first=True
):
    """
    Ajusta (fit) os encoders SOMENTE no treino e devolve:
      - X_train_enc (com time/event)
      - encoders (dict com oe, ohe, metadata)
      - feature_columns (colunas finais do X após encoding)
    """
    df_aux = df_train.copy().reset_index(drop=True)

    # ===== Ordinal =====
    oe = OrdinalEncoder(
        categories=[list(ordinal_order)],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    df_aux[[ordinal_col]] = oe.fit_transform(df_aux[[ordinal_col]])

    # ===== One-Hot =====
    ohe = OneHotEncoder(
        handle_unknown="ignore",
        drop="first" if drop_first else None,
        sparse_output=False
    )

    X_ohe = ohe.fit_transform(df_aux[list(onehot_cols)])
    ohe_cols = ohe.get_feature_names_out(list(onehot_cols))
    df_ohe = pd.DataFrame(X_ohe, columns=ohe_cols, index=df_aux.index)

    df_aux = df_aux.drop(columns=list(onehot_cols))
    X_train_enc = pd.concat([df_aux, df_ohe], axis=1)

    # bool -> int (segurança)
    for c in X_train_enc.columns:
        if X_train_enc[c].dtype == "bool":
            X_train_enc[c] = X_train_enc[c].astype(int)

    encoders = {"ordinal": oe, "onehot": ohe, "ordinal_col": ordinal_col, "onehot_cols": onehot_cols}
    feature_columns = X_train_enc.columns.tolist()

    return X_train_enc, encoders, feature_columns


def transform_with_encoders(df_test, encoders, feature_columns):
    """
    Aplica os encoders do treino no teste (apenas transform).
    Também alinha as colunas finais exatamente como no treino.
    """
    df_aux = df_test.copy().reset_index(drop=True)

    oe = encoders["ordinal"]
    ohe = encoders["onehot"]
    ordinal_col = encoders["ordinal_col"]
    onehot_cols = encoders["onehot_cols"]

    # ===== Ordinal transform =====
    df_aux[[ordinal_col]] = oe.transform(df_aux[[ordinal_col]])

    # ===== One-Hot transform =====
    X_ohe = ohe.transform(df_aux[list(onehot_cols)])
    ohe_cols = ohe.get_feature_names_out(list(onehot_cols))
    df_ohe = pd.DataFrame(X_ohe, columns=ohe_cols, index=df_aux.index)

    df_aux = df_aux.drop(columns=list(onehot_cols))
    X_test_enc = pd.concat([df_aux, df_ohe], axis=1)

    # bool -> int
    for c in X_test_enc.columns:
        if X_test_enc[c].dtype == "bool":
            X_test_enc[c] = X_test_enc[c].astype(int)

    # ===== Alinhar colunas com treino =====
    for col in feature_columns:
        if col not in X_test_enc.columns:
            X_test_enc[col] = 0

    X_test_enc = X_test_enc[feature_columns]

    return X_test_enc


def make_sksurv_xy(X_enc, time_col="time", event_col="event"):
    """
    Converte dataframe codificado (com time/event) em:
      - X: covariáveis
      - y: Surv(dtype=[('event', '?'), ('time', '<f8')])
    """
    X = X_enc.drop(columns=[time_col, event_col]).copy()
    y = Surv.from_dataframe(event_col, time_col, X_enc)
    return X, y


# ============================================================================
# SECTION 2: Metrics & Tau Computation Functions
# ============================================================================

def compute_safe_tau(y_val, method="event_based", safety_factor=0.8, override_tau=None):
    """
    Computa um tau seguro para IPCW baseado no conjunto fornecido.
    
    Descrição:
      Garante que o tau usado em concordance_index_ipcw está dentro do 
      escopo de eventos observados, evitando extrapolação.
      
      POR FAVOR USE: Dentro de cada fold durante CV (y_fold_train)
                      OU na fase final com y_dev (nunca com y_test)
    
    Parâmetros:
      y_val: structured array - Surv array do conjunto (fold train OU dev inteiro)
      method: str - Método de cálculo ("event_based" default)
      safety_factor: float - Fator de segurança (default=0.8)
      override_tau: float, optional - Se fornecido, retorna este valor
    
    Retorno:
      float - Valor de tau seguro para usar em IPCW (em meses)
    """
    if override_tau is not None:
        print(f"override_tau fornecido: {override_tau:.1f} meses")
        return float(override_tau)
    
    event_times = y_val["time"][y_val["event"] == True]
    
    if len(event_times) == 0:
        max_time = float(np.max(y_val["time"]))
        safe_tau = max_time * safety_factor
        print(f"Aviso: Nenhum evento em y_val. Usando tau={safe_tau:.1f}")
        return float(safe_tau)
    
    max_event_time = float(np.max(event_times))
    safe_tau = max_event_time * safety_factor
    
    return float(safe_tau)


def calc_ipcw_cindex(y_train, y_test, risk_scores, tau):
    """
    Calcula o C-index IPCW com um tau específico.
    
    Parâmetros:
      y_train, y_test: structured arrays de sobrevivência
      risk_scores: array de scores de risco
      tau: float (REQUERIDO) - Valor seguro computado de compute_safe_tau()
    
    Retorno:
      float - C-index IPCW
    """
    if tau is None:
        raise ValueError(
            "tau é REQUERIDO e deve ser computado via compute_safe_tau(). "
            "Não são permitidos fallbacks automáticos."
        )

    cindex_ipcw = concordance_index_ipcw(
        y_train,
        y_test,
        risk_scores,
        tau=tau
    )[0]

    return float(cindex_ipcw)


def eval_survival_model_simple(model, X_train, y_train, X_test, y_test, max_time=60, tau=None):
    """
    Avalia modelo de sobrevida com C-index, C-index IPCW, IBS e BS_last.
    
    Parâmetros:
      tau: float (REQUERIDO) - Safe tau para IPCW
    """
    if tau is None:
        raise ValueError(
            "tau é REQUERIDO e deve ser computado via compute_safe_tau(). "
            "Não são permitidos fallbacks automáticos."
        )
    
    # ---- C-index ----
    risk_test = model.predict(X_test)
    cindex = concordance_index_censored(y_test["event"], y_test["time"], risk_test)[0]
    cindex_ipcw = calc_ipcw_cindex(y_train, y_test, risk_test, tau=tau)

    # ---- Curvas e grid de tempos ----
    sfs = model.predict_survival_function(X_test, return_array=False)

    times_full = np.asarray(sfs[0].x, dtype=float)
    mask = (times_full > 0) & (times_full <= max_time)
    times = times_full[mask]

    surv = np.vstack([sf.y[mask] for sf in sfs]).astype(float)

    # ---- IBS (Integrated Brier Score) ----
    ibs_val = integrated_brier_score(y_train, y_test, surv, times)

    # ---- BS (Brier Score at final time point) ----
    bs_times, bs_vals = brier_score(y_train, y_test, surv, times=times)
    bs_last = float(bs_vals[-1])

    return {
        "C-index": float(cindex),
        "C-index IPCW": float(cindex_ipcw),
        "IBS": float(ibs_val),
        "BS_last": bs_last
    }


# ============================================================================
# SECTION 3: K-Fold Cross-Validation Infrastructure
# ============================================================================

def fit_encoders_on_fold(df_fold_train, ordinal_col="ECGRUP", ordinal_order=("I", "II", "III", "IV"), onehot_cols=("TOPO", "MORFO_CAT"), drop_first=True):
    """Fit encoders on fold training data."""
    return fit_encoders_train(df_fold_train, ordinal_col, ordinal_order, onehot_cols, drop_first)


def apply_encoders_to_fold_val(df_fold_val, fold_encoders, feature_columns):
    """Apply fold encoders to fold validation data."""
    return transform_with_encoders(df_fold_val, fold_encoders, feature_columns)


def run_kfold_cv_for_model(
    df_dev,
    model_fit_fn,
    model_eval_fn,
    model_config=None,
    n_splits=5,
    random_state=19,
    verbose=True
):
    """
    Executa k-fold cross-validation no conjunto de desenvolvimento.
    
    CRÍTICO:
      - Encoders fit SOMENTE na porção treino de cada fold
      - safe_tau computado DENTRO de cada fold a partir de y_fold_train
      - y_fold_val NÃO influencia a horizonte IPCW
    
    Parâmetros:
      df_dev: DataFrame de desenvolvimento
      model_fit_fn: função(X_train, y_train, X_val, y_val, config) -> model
      model_eval_fn: função(model, X_train, y_train, X_val, y_val, tau) -> dict de métricas
      model_config: dict com hiperparâmetros ou None
      n_splits: int - número de folds (default=5)
      random_state: int - seed para StratifiedKFold
      verbose: bool - imprimir progresso
    
    Retorno:
      {
        "fold_metrics": [dict, dict, ...],
        "mean_metrics": dict com chaves "C-index", "C-index IPCW", "IBS", "BS_last"
        "std_metrics": dict com mesmas chaves (desvios padrão)
      }
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    df_dev = df_dev.copy().reset_index(drop=True)
    strat_col = df_dev["event"].values
    
    fold_metrics_list = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_dev, strat_col)):
        if verbose:
            print(f"\n--- Fold {fold_idx + 1}/{n_splits} ---")
        
        # 1) Subset para o fold
        df_fold_train = df_dev.iloc[train_idx].reset_index(drop=True)
        df_fold_val = df_dev.iloc[val_idx].reset_index(drop=True)
        
        # 2) Fit encoders SOMENTE on fold train
        X_fold_train_enc, fold_encoders, fold_feature_cols = fit_encoders_on_fold(
            df_fold_train,
            ordinal_col="ECGRUP",
            ordinal_order=("I", "II", "III", "IV"),
            onehot_cols=("TOPO", "MORFO_CAT"),
            drop_first=True
        )
        
        # 3) Transform fold val com fold encoders
        X_fold_val_enc = apply_encoders_to_fold_val(df_fold_val, fold_encoders, fold_feature_cols)
        
        # 4) Converter para Surv format
        X_fold_train, y_fold_train = make_sksurv_xy(X_fold_train_enc, "time", "event")
        X_fold_val, y_fold_val = make_sksurv_xy(X_fold_val_enc, "time", "event")
        
        if verbose:
            print(f"  Train: {X_fold_train.shape}, Val: {X_fold_val.shape}")
        
        # 5) ⭐ CRÍTICO: Compute safe_tau from fold train ONLY ⭐
        safe_tau_fold = compute_safe_tau(y_fold_train)
        
        # 6) Treina modelo
        model = model_fit_fn(X_fold_train, y_fold_train, X_fold_val, y_fold_val, model_config)
        
        # 7) Avalia usando fold-specific tau
        fold_metrics = model_eval_fn(model, X_fold_train, y_fold_train, X_fold_val, y_fold_val, safe_tau_fold)
        fold_metrics_list.append(fold_metrics)
        
        if verbose:
            print(f"  Metrics: C-Index={fold_metrics.get('C-index', np.nan):.4f}, IBS={fold_metrics.get('IBS', np.nan):.4f}, safe_tau_fold={safe_tau_fold:.1f}")
    
    # 8) Agrega
    metric_keys = list(fold_metrics_list[0].keys())
    mean_metrics = {}
    std_metrics = {}
    
    for key in metric_keys:
        values = np.array([m[key] for m in fold_metrics_list])
        mean_metrics[key] = float(np.mean(values))
        std_metrics[key] = float(np.std(values))
    
    if verbose:
        print(f"\n✓ CV completed for all {n_splits} folds")
        print(f"  Mean C-Index: {mean_metrics['C-index']:.4f} ± {std_metrics['C-index']:.4f}")
    
    return {
        "fold_metrics": fold_metrics_list,
        "mean_metrics": mean_metrics,
        "std_metrics": std_metrics
    }


# ============================================================================
# SECTION 4: Model-Specific Fit & Eval Functions
# ============================================================================

# --- RSF (Random Survival Forest) ---
def rsf_fit_fn(X_train, y_train, X_val, y_val, config):
    """Fit function for RSF (no early stopping needed)."""
    model = RandomSurvivalForest(
        n_estimators=config.get("n_estimators", 300) if config else 300,
        min_samples_split=config.get("min_samples_split", 10) if config else 10,
        min_samples_leaf=config.get("min_samples_leaf", 15) if config else 15,
        max_features=config.get("max_features", "sqrt") if config else "sqrt",
        n_jobs=-1,
        random_state=19,
        oob_score=True
    )
    model.fit(X_train, y_train)
    return model


def rsf_eval_fn(model, X_train, y_train, X_val, y_val, tau):
    """Evaluation function for RSF."""
    return eval_survival_model_simple(model, X_train, y_train, X_val, y_val, max_time=60, tau=tau)


# --- GBSA (Gradient Boosting Survival Analysis) ---
def gbsa_fit_fn(X_train, y_train, X_val, y_val, config):
    """Fit function for GBSA."""
    model = GradientBoostingSurvivalAnalysis(
        loss=config.get("loss", "coxph") if config else "coxph",
        learning_rate=config.get("learning_rate", 0.05) if config else 0.05,
        n_estimators=config.get("n_estimators", 300) if config else 300,
        subsample=config.get("subsample", 0.8) if config else 0.8,
        max_depth=config.get("max_depth", 3) if config else 3,
        random_state=19
    )
    model.fit(X_train, y_train)
    return model


def gbsa_eval_fn(model, X_train, y_train, X_val, y_val, tau):
    """Evaluation function for GBSA."""
    return eval_survival_model_simple(model, X_train, y_train, X_val, y_val, max_time=60, tau=tau)


# --- XGBoost Cox ---
def make_xgb_cox_label(y_struct):
    """Convert structured array to XGBoost Cox labels (censored -> negative)."""
    t = y_struct["time"].astype(float)
    e = y_struct["event"].astype(bool)
    return np.where(e, t, -t)


def xgboost_cox_fit_fn(X_train, y_train, X_val, y_val, config):
    """Fit function for XGBoost Cox with early stopping on validation set."""
    dtrain = xgb.DMatrix(X_train, label=make_xgb_cox_label(y_train))
    dval = xgb.DMatrix(X_val, label=make_xgb_cox_label(y_val))
    
    params = config if config else {
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 4,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "seed": 42,
    }
    
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dtrain, "train"), (dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    return bst


def xgb_cox_survival_curves_via_lifelines(bst, X_train, y_train, X_test, max_time=60, penalizer=1e-3):
    """Generate survival curves for XGBoost Cox via post-hoc calibration."""
    dtrain = xgb.DMatrix(X_train)
    dtest = xgb.DMatrix(X_test)
    eta_train = bst.predict(dtrain)
    eta_test = bst.predict(dtest)

    df_fit = pd.DataFrame({
        "eta": eta_train,
        "time": y_train["time"].astype(float),
        "event": y_train["event"].astype(int),
    })

    cph_eta = CoxPHFitter(penalizer=penalizer)
    cph_eta.fit(df_fit, duration_col="time", event_col="event")

    df_test_eta = pd.DataFrame({"eta": eta_test})
    sf = cph_eta.predict_survival_function(df_test_eta)

    times = sf.index.values
    times = times[(times > 0) & (times <= max_time)]
    sf = sf.loc[times]
    surv = sf.T.values

    return times, surv


def xgboost_cox_eval_fn(model, X_train, y_train, X_val, y_val, tau):
    """Evaluation function for XGBoost Cox."""
    risk_test = model.predict(xgb.DMatrix(X_val))
    cindex = concordance_index_censored(y_val["event"], y_val["time"], risk_test)[0]
    cindex_ipcw = calc_ipcw_cindex(y_train, y_val, risk_test, tau=tau)

    times, surv = xgb_cox_survival_curves_via_lifelines(model, X_train, y_train, X_val, max_time=60)
    
    ibs_val = integrated_brier_score(y_train, y_val, surv, times)
    bs_times, bs_vals = brier_score(y_train, y_val, surv, times=times)

    return {
        "C-index": float(cindex),
        "C-index IPCW": float(cindex_ipcw),
        "IBS": float(ibs_val),
        "BS_last": float(bs_vals[-1])
    }


# --- XGBoost AFT ---
def make_aft_bounds(y_struct):
    """Convert structured array to AFT bounds (censored -> [t, inf])."""
    t = y_struct["time"].astype(float)
    e = y_struct["event"].astype(bool)
    lower = t.copy()
    upper = np.where(e, t, np.inf)
    return lower, upper


def make_dmatrix_aft(X, y_struct):
    """Create DMatrix with AFT bounds."""
    dmat = xgb.DMatrix(X)
    lower, upper = make_aft_bounds(y_struct)
    dmat.set_float_info("label_lower_bound", lower)
    dmat.set_float_info("label_upper_bound", upper)
    return dmat


def xgboost_aft_fit_fn(X_train, y_train, X_val, y_val, config):
    """Fit function for XGBoost AFT with early stopping."""
    dtrain = make_dmatrix_aft(X_train, y_train)
    dval = make_dmatrix_aft(X_val, y_val)
    
    params = config if config else {
        "objective": "survival:aft",
        "eval_metric": "aft-nloglik",
        "aft_loss_distribution": "normal",
        "aft_loss_distribution_scale": 5.0,
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 50,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 5.0,
        "alpha": 0.0,
        "seed": 42,
    }
    
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=6000,
        evals=[(dtrain, "train"), (dval, "validation")],
        early_stopping_rounds=200,
        verbose_eval=False
    )
    return bst


def aft_survival_curves_via_lifelines(bst_aft, X_train, y_train, X_test, max_time=60, penalizer=1e-3):
    """Generate survival curves for XGBoost AFT via post-hoc calibration."""
    mu_train = bst_aft.predict(xgb.DMatrix(X_train))
    mu_test = bst_aft.predict(xgb.DMatrix(X_test))

    risk_train = -mu_train
    risk_test = -mu_test

    df_fit = pd.DataFrame({
        "risk": risk_train.astype(float),
        "time": y_train["time"].astype(float),
        "event": y_train["event"].astype(int),
    })

    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(df_fit, duration_col="time", event_col="event")

    df_test = pd.DataFrame({"risk": risk_test.astype(float)})
    sf = cph.predict_survival_function(df_test)

    times = sf.index.values.astype(float)
    mask = (times > 0) & (times <= max_time)
    times = times[mask]
    surv = sf.loc[times].T.values.astype(float)

    return times, surv, risk_test


def xgboost_aft_eval_fn(model, X_train, y_train, X_val, y_val, tau):
    """Evaluation function for XGBoost AFT."""
    risk_test = -model.predict(xgb.DMatrix(X_val))
    cindex = concordance_index_censored(y_val["event"], y_val["time"], risk_test)[0]
    cindex_ipcw = calc_ipcw_cindex(y_train, y_val, risk_test, tau=tau)

    times, surv, _ = aft_survival_curves_via_lifelines(
        model, X_train, y_train, X_val, max_time=60
    )

    ibs_val = integrated_brier_score(y_train, y_val, surv, times)
    _, bs_vals = brier_score(y_train, y_val, surv, times=times)

    return {
        "C-index": float(cindex),
        "C-index IPCW": float(cindex_ipcw),
        "IBS": float(ibs_val),
        "BS_last": float(bs_vals[-1])
    }


# --- LightGBM Cox ---
def _cox_group_denoms(time_sorted_desc, exp_f_sorted_desc):
    """Breslow ties handling for Cox."""
    cum_risk = np.cumsum(exp_f_sorted_desc)
    t = time_sorted_desc
    is_end = np.r_[t[1:] != t[:-1], True]
    group_end = np.where(is_end)[0]

    denom = np.empty_like(cum_risk)
    start = 0
    for end in group_end:
        denom_val = cum_risk[end]
        denom[start:end + 1] = denom_val
        start = end + 1
    return denom


def cox_negloglik_and_gradhess(pred, time, event):
    """Gradient and Hessian for Cox PH (Breslow ties)."""
    time = time.astype(float)
    event = event.astype(np.int8)

    order = np.argsort(-time)
    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))

    t_s = time[order]
    e_s = event[order]
    f_s = pred[order].astype(float)

    f_s = np.clip(f_s, -50.0, 50.0)
    exp_f = np.exp(f_s)

    denom = _cox_group_denoms(t_s, exp_f)
    denom = np.clip(denom, 1e-12, None)

    w = e_s / denom
    u = e_s / (denom ** 2)

    S = np.cumsum(w[::-1])[::-1]
    T = np.cumsum(u[::-1])[::-1]

    grad_s = exp_f * S - e_s
    hess_s = exp_f * S - (exp_f ** 2) * T
    hess_s = np.clip(hess_s, 1e-12, None)

    grad = grad_s[inv]
    hess = hess_s[inv]
    return grad, hess


def cox_negloglik_metric(pred, time, event):
    """Negative partial log-likelihood metric (Breslow ties)."""
    time = time.astype(float)
    event = event.astype(np.int8)

    order = np.argsort(-time)
    t_s = time[order]
    e_s = event[order]
    f_s = np.clip(pred[order].astype(float), -50.0, 50.0)

    exp_f = np.exp(f_s)
    denom = _cox_group_denoms(t_s, exp_f)
    denom = np.clip(denom, 1e-12, None)

    val = -np.sum(e_s * (f_s - np.log(denom)))
    n_events = max(1, int(e_s.sum()))
    return float(val / n_events)


def lgbm_cox_fit_fn(X_train, y_train, X_val, y_val, config):
    """Fit function for LightGBM Cox with custom objective and early stopping."""
    time_tr = y_train["time"].astype(float)
    event_tr = y_train["event"].astype(bool).astype(np.int8)
    time_val = y_val["time"].astype(float)
    event_val = y_val["event"].astype(bool).astype(np.int8)

    # Standardize column names for LightGBM
    X_train_std = X_train.copy()
    X_val_std = X_val.copy()
    X_train_std.columns = X_train_std.columns.astype(str).str.replace(r"\s+", "_", regex=True)
    X_val_std.columns = X_val_std.columns.astype(str).str.replace(r"\s+", "_", regex=True)

    if config is None:
        config = dict(
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=50,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            max_depth=-1,
            random_state=19,
            n_estimators=4000
        )

    def custom_objective(y_true_time, y_pred):
        grad, hess = cox_negloglik_and_gradhess(y_pred, y_true_time, event_tr)
        return grad, hess

    def custom_metric(y_true_time, y_pred):
        if len(y_true_time) == len(time_val):
            e = event_val
        else:
            e = event_tr
        val = cox_negloglik_metric(y_pred, y_true_time, e)
        return ("cox-nloglik", val, False)

    model = lgb.LGBMRegressor(objective=custom_objective, **config)
    
    model.fit(
        X_train_std, time_tr,
        eval_set=[(X_train_std, time_tr), (X_val_std, time_val)],
        eval_names=["train", "validation"],
        eval_metric=custom_metric,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=500),
        ],
    )
    
    model._original_cols = X_train.columns.tolist()
    return model


def lgbm_cox_survival_curves_via_lifelines(model, X_train, y_train, X_test, max_time=60, penalizer=1e-3):
    """Generate survival curves for LightGBM Cox via post-hoc calibration."""
    risk_train = model.predict(X_train).astype(float)
    risk_test = model.predict(X_test).astype(float)

    df_fit = pd.DataFrame({
        "risk": risk_train,
        "time": y_train["time"].astype(float),
        "event": y_train["event"].astype(int),
    })

    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(df_fit, duration_col="time", event_col="event")

    df_test = pd.DataFrame({"risk": risk_test})
    sf = cph.predict_survival_function(df_test)

    times = sf.index.values.astype(float)
    mask = (times > 0) & (times <= max_time)
    times = times[mask]
    surv = sf.loc[times].T.values.astype(float)

    return times, surv, risk_test


def lgbm_cox_eval_fn(model, X_train, y_train, X_val, y_val, tau):
    """Evaluation function for LightGBM Cox."""
    risk_test = model.predict(X_val)
    cindex = concordance_index_censored(y_val["event"], y_val["time"], risk_test)[0]
    cindex_ipcw = calc_ipcw_cindex(y_train, y_val, risk_test, tau=tau)

    times, surv, _ = lgbm_cox_survival_curves_via_lifelines(
        model, X_train, y_train, X_val, max_time=60
    )

    ibs_val = integrated_brier_score(y_train, y_val, surv, times)
    bs_times, bs_vals = brier_score(y_train, y_val, surv, times=times)

    return {
        "C-index": float(cindex),
        "C-index IPCW": float(cindex_ipcw),
        "IBS": float(ibs_val),
        "BS_last": float(bs_vals[-1])
    }


# ============================================================================
# SECTION 5: Utilities & Plotting
# ============================================================================

def results_table(results_dict, sort_by="C-index", ascending=False):
    """
    Consolida resultados de múltiplos modelos em um DataFrame ordenado.
    """
    df = pd.DataFrame(results_dict).T

    bs_column = None
    if "BS_last" in df.columns:
        bs_column = "BS_last"
    else:
        for col in df.columns:
            if col.startswith('BS@'):
                bs_column = col
                break
    
    ordered_cols = [c for c in ["C-index", "C-index IPCW", "IBS"] if c in df.columns]
    if bs_column:
        ordered_cols.append(bs_column)
    ordered_cols += [c for c in df.columns if c not in ordered_cols]
    
    df = df[ordered_cols]
    df = df.sort_values(by=sort_by, ascending=ascending)
    return df

print("✓ Phase 3: All helper functions consolidated and ready.")

✓ Phase 3: All helper functions consolidated and ready.


In [4]:
df_colo = pd.read_csv('../DataSet/Colorretal_tratado_2025.csv', index_col=0)
df_colo

,ESCOLARI,IDADE,SEXO,CATEATEND,DIAGPREV,TOPO,ECGRUP,ANODIAG,HABILIT,nDRS,CONSDIAG_CAT,TRATCONS_CAT,DIAGTRAT_CAT,MORFO_CAT,time,event
INSTITU,,,,,,,,,,,,,,,,
9326,2,20,1,2,1,C182,IV,2019,1,9,2,2,0,Adenocarcinoma,33,1
12,4,20,2,9,2,C209,IV,2008,2,7,3,0,0,Adenocarcinoma,4,1
16624,5,20,2,2,2,C189,IV,2014,1,1,3,2,2,Adenocarcinoma,26,1
8672,4,20,2,2,1,C209,IV,2015,1,1,0,1,0,Adenocarcinoma,3,1
20737,4,20,2,2,2,C187,III,2016,1,2,3,0,0,Adenocarcinoma,61,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
612374,9,97,2,2,2,C209,III,2016,1,1,3,0,0,Adenocarcinoma,9,1
612374,9,99,2,2,2,C209,IV,2013,1,1,3,1,2,Adenocarcinoma,19,1
20621,2,99,1,2,2,C209,I,2016,2,3,3,0,0,Adenocarcinoma,17,1


In [5]:
display(df_colo["time"])

INSTITU
9326      33
12         4
16624     26
8672       3
20737     61
          ..
612374     9
612374    19
20621     17
208066     2
20141      7
Name: time, Length: 32664, dtype: int64

### Data Splitting Strategy

Two-part split: Development set (85%) for model development, External Test set (15%) for final unbiased evaluation.


### Encoders

In [6]:
# === TWO-PART SPLIT: Development (85%) and External Test (15%) ===
# The test set is LOCKED and will only be touched for final evaluation.
# All model selection, hyperparameter tuning, and validation happens in the development set.

df_dev, df_test_external = split_dev_test(
    df_colo, 
    dev_ratio=0.85, 
    test_ratio=0.15,
    random_state=19, 
    stratify_col="event"
)

print(f"Dados originais: {df_colo.shape}")
print(f"Desenvolvimento (85%): {df_dev.shape}")
print(f"Teste Externo (15%): {df_test_external.shape}") 
print(f"\nTotal: {df_dev.shape[0] + df_test_external.shape[0]}")
print(f"\n🔒 TEST SET LOCKED: df_test_external is protected until final evaluation phase.")

# 2) Fit encoders on development set (NOT on test set)
X_dev_enc, encoders_ml, feature_cols = fit_encoders_train(
    df_dev,
    ordinal_col="ECGRUP",
    ordinal_order=("I", "II", "III", "IV"),
    onehot_cols=("TOPO", "MORFO_CAT"),
    drop_first=True
)

# 3) Apply encoders to external test set (never refit)
X_test_external_enc = transform_with_encoders(df_test_external, encoders_ml, feature_cols)

print("\nFormas após encoding:")
print(f"  Desenvolvimento:   {X_dev_enc.shape}")
print(f"  Teste Externo:     {X_test_external_enc.shape}")


Dados originais: (32664, 16)
Desenvolvimento (85%): (27764, 16)
Teste Externo (15%): (4900, 16)

Total: 32664

🔒 TEST SET LOCKED: df_test_external is protected until final evaluation phase.

Formas após encoding:
  Desenvolvimento:   (27764, 28)
  Teste Externo:     (4900, 28)


In [7]:
# === Validate encoder alignment between development and external test sets ===
assert X_dev_enc.shape[1] == X_test_external_enc.shape[1], \
    f"Feature count mismatch: dev={X_dev_enc.shape[1]}, test_external={X_test_external_enc.shape[1]}"
assert list(X_dev_enc.columns) == list(X_test_external_enc.columns), \
    "Column names don't match between dev and test_external sets"
print(f"✓ Encoder alignment verified: {X_dev_enc.shape[1]} features across dev/test_external sets")


✓ Encoder alignment verified: 28 features across dev/test_external sets


### Divisão em X e y

In [8]:
# ============================================================================
# PHASE 5: K-FOLD CROSS-VALIDATION ON DEVELOPMENT SET
# ============================================================================
# Each model runs 5-fold CV on the full development set.
# Encoders fit only on each fold's training data (no data leakage).
# safe_tau computed per-fold from fold training data only (IPCW horizon isolation).
# External test set remains locked and untouched.

cv_results_dict = {}

# === RSF Cross-Validation ===
print("\n" + "="*80)
print("PHASE 5A: Random Survival Forest (k-fold CV on Development Set)")
print("="*80)

rsf_cv_results = run_kfold_cv_for_model(
    df_dev,
    rsf_fit_fn,
    rsf_eval_fn,
    model_config={
        "n_estimators": 300,
        "min_samples_split": 10,
        "min_samples_leaf": 15,
        "max_features": "sqrt"
    },
    n_splits=5,
    random_state=19,
    verbose=True
)
cv_results_dict["RSF"] = rsf_cv_results

print("\n✓ RSF CV Results:")
print(f"  C-Index: {rsf_cv_results['mean_metrics']['C-index']:.4f} ± {rsf_cv_results['std_metrics']['C-index']:.4f}")
print(f"  IBS:     {rsf_cv_results['mean_metrics']['IBS']:.4f} ± {rsf_cv_results['std_metrics']['IBS']:.4f}")



PHASE 5A: Random Survival Forest (k-fold CV on Development Set)

--- Fold 1/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7400, IBS=0.1598, safe_tau_fold=48.0

--- Fold 2/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7417, IBS=0.1592, safe_tau_fold=48.0

--- Fold 3/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7429, IBS=0.1576, safe_tau_fold=48.0

--- Fold 4/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7487, IBS=0.1560, safe_tau_fold=48.0

--- Fold 5/5 ---
  Train: (22212, 26), Val: (5552, 26)
  Metrics: C-Index=0.7347, IBS=0.1621, safe_tau_fold=48.0

✓ CV completed for all 5 folds
  Mean C-Index: 0.7416 ± 0.0045

✓ RSF CV Results:
  C-Index: 0.7416 ± 0.0045
  IBS:     0.1589 ± 0.0020


In [9]:
# === GBSA (Gradient Boosting Survival Analysis) Cross-Validation ===
print("\n" + "="*80)
print("PHASE 5B: Gradient Boosting Survival Analysis (k-fold CV on Development Set)")
print("="*80)

gbsa_cv_results = run_kfold_cv_for_model(
    df_dev,
    gbsa_fit_fn,
    gbsa_eval_fn,
    model_config={
        "loss": "coxph",
        "learning_rate": 0.05,
        "n_estimators": 300,
        "subsample": 0.8,
        "max_depth": 3
    },
    n_splits=5,
    random_state=19,
    verbose=True
)
cv_results_dict["GBSA"] = gbsa_cv_results

print("\n✓ GBSA CV Results:")
print(f"  C-Index: {gbsa_cv_results['mean_metrics']['C-index']:.4f} ± {gbsa_cv_results['std_metrics']['C-index']:.4f}")
print(f"  IBS:     {gbsa_cv_results['mean_metrics']['IBS']:.4f} ± {gbsa_cv_results['std_metrics']['IBS']:.4f}")


PHASE 5B: Gradient Boosting Survival Analysis (k-fold CV on Development Set)

--- Fold 1/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7366, IBS=0.1611, safe_tau_fold=48.0

--- Fold 2/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7372, IBS=0.1610, safe_tau_fold=48.0

--- Fold 3/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7431, IBS=0.1582, safe_tau_fold=48.0

--- Fold 4/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7464, IBS=0.1569, safe_tau_fold=48.0

--- Fold 5/5 ---
  Train: (22212, 26), Val: (5552, 26)
  Metrics: C-Index=0.7308, IBS=0.1636, safe_tau_fold=48.0

✓ CV completed for all 5 folds
  Mean C-Index: 0.7388 ± 0.0054

✓ GBSA CV Results:
  C-Index: 0.7388 ± 0.0054
  IBS:     0.1602 ± 0.0024


In [10]:
# === XGBoost Cox Cross-Validation ===
print("\n" + "="*80)
print("PHASE 5C: XGBoost Cox (k-fold CV on Development Set)")
print("="*80)

xgb_cox_cv_results = run_kfold_cv_for_model(
    df_dev,
    xgboost_cox_fit_fn,
    xgboost_cox_eval_fn,
    model_config={
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 4,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "seed": 42,
    },
    n_splits=5,
    random_state=19,
    verbose=True
)
cv_results_dict["XGBoost Cox"] = xgb_cox_cv_results

print("\n✓ XGBoost Cox CV Results:")
print(f"  C-Index: {xgb_cox_cv_results['mean_metrics']['C-index']:.4f} ± {xgb_cox_cv_results['std_metrics']['C-index']:.4f}")
print(f"  IBS:     {xgb_cox_cv_results['mean_metrics']['IBS']:.4f} ± {xgb_cox_cv_results['std_metrics']['IBS']:.4f}")


PHASE 5C: XGBoost Cox (k-fold CV on Development Set)

--- Fold 1/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7407, IBS=0.1960, safe_tau_fold=48.0

--- Fold 2/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7430, IBS=0.1916, safe_tau_fold=48.0

--- Fold 3/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7472, IBS=0.1888, safe_tau_fold=48.0

--- Fold 4/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7514, IBS=0.1884, safe_tau_fold=48.0

--- Fold 5/5 ---
  Train: (22212, 26), Val: (5552, 26)
  Metrics: C-Index=0.7356, IBS=0.1936, safe_tau_fold=48.0

✓ CV completed for all 5 folds
  Mean C-Index: 0.7436 ± 0.0054

✓ XGBoost Cox CV Results:
  C-Index: 0.7436 ± 0.0054
  IBS:     0.1917 ± 0.0029


In [11]:
# === XGBoost AFT Cross-Validation ===
print("\n" + "="*80)
print("PHASE 5D: XGBoost AFT (k-fold CV on Development Set)")
print("="*80)

xgb_aft_cv_results = run_kfold_cv_for_model(
    df_dev,
    xgboost_aft_fit_fn,
    xgboost_aft_eval_fn,
    model_config={
        "objective": "survival:aft",
        "eval_metric": "aft-nloglik",
        "aft_loss_distribution": "normal",
        "aft_loss_distribution_scale": 5.0,
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 50,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 5.0,
        "alpha": 0.0,
        "seed": 42,
    },
    n_splits=5,
    random_state=19,
    verbose=True
)
cv_results_dict["XGBoost AFT"] = xgb_aft_cv_results

print("\n✓ XGBoost AFT CV Results:")
print(f"  C-Index: {xgb_aft_cv_results['mean_metrics']['C-index']:.4f} ± {xgb_aft_cv_results['std_metrics']['C-index']:.4f}")
print(f"  IBS:     {xgb_aft_cv_results['mean_metrics']['IBS']:.4f} ± {xgb_aft_cv_results['std_metrics']['IBS']:.4f}")



PHASE 5D: XGBoost AFT (k-fold CV on Development Set)

--- Fold 1/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7278, IBS=0.1901, safe_tau_fold=48.0

--- Fold 2/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7312, IBS=0.1899, safe_tau_fold=48.0

--- Fold 3/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7351, IBS=0.1861, safe_tau_fold=48.0

--- Fold 4/5 ---
  Train: (22211, 26), Val: (5553, 26)
  Metrics: C-Index=0.7379, IBS=0.1870, safe_tau_fold=48.0

--- Fold 5/5 ---
  Train: (22212, 26), Val: (5552, 26)
  Metrics: C-Index=0.7192, IBS=0.1936, safe_tau_fold=48.0

✓ CV completed for all 5 folds
  Mean C-Index: 0.7303 ± 0.0065

✓ XGBoost AFT CV Results:
  C-Index: 0.7303 ± 0.0065
  IBS:     0.1893 ± 0.0026


In [12]:
# === LightGBM Cox Cross-Validation ===
print("\n" + "="*80)
print("PHASE 5E: LightGBM Cox (k-fold CV on Development Set)")
print("="*80)

lgbm_cox_cv_results = run_kfold_cv_for_model(
    df_dev,
    lgbm_cox_fit_fn,
    lgbm_cox_eval_fn,
    model_config=dict(
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        max_depth=-1,
        random_state=19,
        n_estimators=4000
    ),
    n_splits=5,
    random_state=19,
    verbose=True
)
cv_results_dict["LightGBM Cox"] = lgbm_cox_cv_results

print("\n✓ LightGBM Cox CV Results:")
print(f"  C-Index: {lgbm_cox_cv_results['mean_metrics']['C-index']:.4f} ± {lgbm_cox_cv_results['std_metrics']['C-index']:.4f}")
print(f"  IBS:     {lgbm_cox_cv_results['mean_metrics']['IBS']:.4f} ± {lgbm_cox_cv_results['std_metrics']['IBS']:.4f}")

print("\n" + "="*80)
print("✓ PHASE 5 COMPLETE: All 5 models evaluated via 5-fold CV on development set")
print("="*80)



PHASE 5E: LightGBM Cox (k-fold CV on Development Set)

--- Fold 1/5 ---
  Train: (22211, 26), Val: (5553, 26)
[LightGBM] [Info] Using self-defined objective function
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 181
[LightGBM] [Info] Number of data points in the train set: 22211, number of used features: 26
[LightGBM] [Info] Using self-defined objective function
  Metrics: C-Index=0.7211, IBS=0.1807, safe_tau_fold=48.0

--- Fold 2/5 ---
  Train: (22211, 26), Val: (5553, 26)
[LightGBM] [Info] Using self-defined objective function
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002454 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 181
[LightGBM] [Info] Number of data points in the train set: 22

## RSF

## K-Fold Cross-Validation Infrastructure

All model development now uses k-fold cross-validation on the development set.
Each fold maintains proper data separation: encoders fit only on fold-training, transform applied to fold-validation.
Metrics are aggregated (mean ± std) across folds for robust model comparison.



## Gradient Boosting Survival

## XGBoost Cox

## XGBoost AFT (CV)


## LightGBM Cox (CV)


## Phase 5: Cross-Validation Results Summary

Aggregate results from all k-fold CV runs. Each model's performance is reported as mean ± std across folds.
This table guides model selection and hyperparameter tuning decisions.



In [13]:
# ============================================================================
# PHASE 6: CROSS-VALIDATION RESULTS SUMMARY
# ============================================================================

# Create summary dataframe from cv_results_dict
summary_rows = []
for model_name, cv_results in cv_results_dict.items():
    mean_metrics = cv_results["mean_metrics"]
    std_metrics = cv_results["std_metrics"]
    
    summary_rows.append({
        "Model": model_name,
        "C-Index": f"{mean_metrics['C-index']:.4f} ± {std_metrics['C-index']:.4f}",
        "C-Index IPCW": f"{mean_metrics['C-index IPCW']:.4f} ± {std_metrics['C-index IPCW']:.4f}",
        "IBS": f"{mean_metrics['IBS']:.4f} ± {std_metrics['IBS']:.4f}",
        "BS_last": f"{mean_metrics['BS_last']:.4f} ± {std_metrics['BS_last']:.4f}",
    })

cv_summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*120)
print("PHASE 6: CROSS-VALIDATION RESULTS SUMMARY (Mean ± Std across 5 folds)")
print("="*120)
print(cv_summary_df.to_string(index=False))
print("="*120)
print("\n📊 Interpretation Guide:")
print("  • Each metric shows: mean ± standard deviation across 5 folds")
print("  • Stable models have low std (minimal variation between folds)")
print("  • C-Index: Concordance (0.5=random, 1.0=perfect)")
print("  • IBS: Integrated Brier Score (lower=better)")
print("  • The best CV performers will be retrained on full dev set for external test evaluation")

# Store for later comparison with external test results
cv_summary = cv_summary_df.copy()



PHASE 6: CROSS-VALIDATION RESULTS SUMMARY (Mean ± Std across 5 folds)
       Model         C-Index    C-Index IPCW             IBS         BS_last
         RSF 0.7416 ± 0.0045 0.7445 ± 0.0044 0.1589 ± 0.0020 0.1763 ± 0.0015
        GBSA 0.7388 ± 0.0054 0.7414 ± 0.0054 0.1602 ± 0.0024 0.1774 ± 0.0016
 XGBoost Cox 0.7436 ± 0.0054 0.7463 ± 0.0054 0.1917 ± 0.0029 0.2180 ± 0.0026
 XGBoost AFT 0.7303 ± 0.0065 0.7326 ± 0.0065 0.1893 ± 0.0026 0.2044 ± 0.0030
LightGBM Cox 0.7220 ± 0.0055 0.7243 ± 0.0055 0.1803 ± 0.0026 0.2031 ± 0.0016

📊 Interpretation Guide:
  • Each metric shows: mean ± standard deviation across 5 folds
  • Stable models have low std (minimal variation between folds)
  • C-Index: Concordance (0.5=random, 1.0=perfect)
  • IBS: Integrated Brier Score (lower=better)
  • The best CV performers will be retrained on full dev set for external test evaluation


In [20]:
print("\n" + "="*80)
print("PHASE 7: FINAL EXTERNAL TEST EVALUATION")
print("🔓 TEST SET UNLOCKED - Evaluating on external test set for unbiased benchmarking")
print("="*80)

# === Encode both dev and test sets using encoders fitted on development data ===
print("\nEncoding development and external test sets...")

X_dev_enc, final_encoders, final_feature_cols = fit_encoders_train(
    df_dev,
    ordinal_col="ECGRUP",
    ordinal_order=("I", "II", "III", "IV"),
    onehot_cols=("TOPO", "MORFO_CAT"),
    drop_first=True
)

# Now encode external test using these encoders
X_test_external_enc = transform_with_encoders(df_test_external, final_encoders, final_feature_cols)

# Convert to Surv format
X_dev_final, y_dev_final = make_sksurv_xy(X_dev_enc, "time", "event")
X_test_external_final, y_test_external_final = make_sksurv_xy(X_test_external_enc, "time", "event")

print(f"✓ Features prepared:")
print(f"  Development set: {X_dev_final.shape[0]} samples, {X_dev_final.shape[1]} features")
print(f"✓ External test set: {X_test_external_final.shape[0]} samples, {X_test_external_final.shape[1]} features")

# === Compute global safe_tau from FULL development set for external test evaluation ===
safe_tau_final = compute_safe_tau(y_dev_final)
print(f"✓ safe_tau for external test evaluation: {safe_tau_final:.1f} months")

# === Train final models on ENTIRE development set and evaluate on external test ===
external_test_results = {}

print("\n[1/5] Training RSF on full development set for external test evaluation...")
rsf_final = RandomSurvivalForest(
    n_estimators=300,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=19,
    oob_score=True
)
rsf_final.fit(X_dev_final, y_dev_final)
rsf_test_metrics = eval_survival_model_simple(rsf_final, X_dev_final, y_dev_final, X_test_external_final, y_test_external_final, max_time=60, tau=safe_tau_final)


PHASE 7: FINAL EXTERNAL TEST EVALUATION
🔓 TEST SET UNLOCKED - Evaluating on external test set for unbiased benchmarking

Encoding development and external test sets...
✓ Features prepared:
  Development set: 27764 samples, 26 features
✓ External test set: 4900 samples, 26 features
✓ safe_tau for external test evaluation: 48.0 months

[1/5] Training RSF on full development set for external test evaluation...


In [16]:
# ============================================================================
# PHASE 8: FINAL RESULTS & GENERALIZATION ASSESSMENT
# ============================================================================

print("\n" + "="*140)
print("PHASE 8: FINAL BENCHMARKING - Cross-Validation vs External Test Set")
print("="*140)

# Create comparison table
comparison_rows = []
for model_name in external_test_results.keys():
    # Get CV results (mean metrics only)
    if model_name in cv_results_dict:
        cv_metrics = cv_results_dict[model_name]["mean_metrics"]
    else:
        cv_metrics = {}
    
    # Get external test results
    test_metrics = external_test_results[model_name]
    
    # Compute delta (difference)
    c_index_delta = test_metrics.get('C-index', np.nan) - cv_metrics.get('C-index', np.nan)
    ibs_delta = test_metrics.get('IBS', np.nan) - cv_metrics.get('IBS', np.nan)
    
    comparison_rows.append({
        "Model": model_name,
        "CV C-Index": f"{cv_metrics.get('C-index', np.nan):.4f}",
        "Test C-Index": f"{test_metrics.get('C-index', np.nan):.4f}",
        "Δ C-Index": f"{c_index_delta:+.4f}",
        "CV IBS": f"{cv_metrics.get('IBS', np.nan):.4f}",
        "Test IBS": f"{test_metrics.get('IBS', np.nan):.4f}",
        "Δ IBS": f"{ibs_delta:+.4f}",
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
print("="*140)

print("\n📊 Generalization Interpretation Guide:")
print("  • Δ (Delta) = External Test Result - CV Result")
print("  • Small negative Δ C-Index → good generalization (stable performance)")
print("  • Large negative Δ C-Index → possible overfitting during CV")
print("  • Small positive Δ C-Index → model benefited from larger training set")
print("  • Δ IBS close to 0 → excellent generalization (predictions remain well-calibrated)")



PHASE 8: FINAL BENCHMARKING - Cross-Validation vs External Test Set
Empty DataFrame
Columns: []
Index: []

📊 Generalization Interpretation Guide:
  • Δ (Delta) = External Test Result - CV Result
  • Small negative Δ C-Index → good generalization (stable performance)
  • Large negative Δ C-Index → possible overfitting during CV
  • Small positive Δ C-Index → model benefited from larger training set
  • Δ IBS close to 0 → excellent generalization (predictions remain well-calibrated)


## Phase 6: Final External Test Evaluation

🔓 **TEST SET NOW UNLOCKED**

Based on CV results, train final models on the entire development set and evaluate once on the external test set.
This provides an unbiased benchmark to assess how models generalize to unseen data.



## Resultados

In [17]:
def results_table(results_dict, sort_by="C-index", ascending=False):
    """
    Consolida resultados de múltiplos modelos em um DataFrame ordenado.
    
    Parâmetros
    ----------
    results_dict : dict
        Dicionário com formato:
        {
            "RSF": {"C-index": ..., "IBS": ..., "BS_last": ...},
            "XGB-Cox": {...},
            ...
        }

    sort_by : str
        Métrica usada para ordenação (default = "C-index")

    ascending : bool
        Ordem crescente ou decrescente

    Retorno
    -------
    pandas.DataFrame
        
    Nota: A métrica Brier Score é armazenada como "BS_last" (valor final, não específico ao tau).
    """
    df = pd.DataFrame(results_dict).T

    # Encontra dinamicamente qual coluna de BS existe
    # Primeiro tenta BS_last (novo padrão), depois BS@*  para compatibilidade com versões antigas
    bs_column = None
    
    if "BS_last" in df.columns:
        bs_column = "BS_last"
    else:
        for col in df.columns:
            if col.startswith('BS@'):
                bs_column = col
                break
    
    # Garante ordem padrão das colunas
    ordered_cols = [c for c in ["C-index", "C-index IPCW", "IBS"] if c in df.columns]
    if bs_column:
        ordered_cols.append(bs_column)
    # Adiciona qualquer coluna restante não prevista
    ordered_cols += [c for c in df.columns if c not in ordered_cols]
    
    df = df[ordered_cols]

    df = df.sort_values(by=sort_by, ascending=ascending)
    return df


In [19]:
# Display combined CV and External Test results
print("\n" + "="*80)
print("📊 EXTERNAL TEST SET RESULTS")
print("="*80)

if external_test_results:
    print("\nModels evaluated on external test set (15% locked data):")
    print("-" * 80)
    for model_name, metrics in external_test_results.items():
        print(f"\n{model_name}:")
        for metric_name, value in metrics.items():
            if isinstance(value, (int, float)):
                print(f"  {metric_name}: {value:.4f}")
            else:
                print(f"  {metric_name}: {value}")
else:
    print("\n⚠️  No external test results available yet.")
    print("Please ensure the external test evaluation cell (Phase 7) has completed successfully.")


📊 EXTERNAL TEST SET RESULTS

⚠️  No external test results available yet.
Please ensure the external test evaluation cell (Phase 7) has completed successfully.
